<a href="https://colab.research.google.com/github/mohamed2ayman/Claude/blob/main/PromptEvaluationDataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install dependencies
%pip install anthropic python-dotenv

#Load new Variable
from dotenv import load_dotenv
load_dotenv()

# Create API Client
import os
from anthropic import Anthropic

from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get('ANTHROPIC_API_KEY')

client = Anthropic()
model = "claude-sonnet-4-5"

In [67]:
# Add user message, while will tak list of messages and text. It's reason is to append messages together and save it to history.

def add_user_message(messages, text):
  user_message = {"role":"user", "content": text}
  messages.append(user_message)

def add_assistant_message(messages, text):
  assistant_message = {"role": "assistant", "content": text}
  messages.append(assistant_message)


def chat(messages, system=None, temperature =1.0, stop_sequences=None ):
  params = {
      "model": model,
      "max_tokens": 1000,
      "messages": messages,
      "temperature": temperature
  }

  if system:
    params["system"] = system

  if stop_sequences:
    params["stop_sequences"] = stop_sequences

  message = client.messages.create(**params)

  return message.content[0].text

In [68]:
# Creating an Evaluation Dataset for Prompt Evaluation

import json

def generate_dataset():
  prompt= """

Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python,
JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python,
JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

  messages = []
  add_user_message(messages, prompt)
  add_assistant_message(messages, "'''json")
  text = chat(messages, stop_sequences=["'''"])
  return json.loads(text)


In [69]:
# Dumping the dataset evaluation in a JSON file
import json
dataset= generate_dataset()

with open("dataset.json", "w") as f:
  json.dump(dataset, f, indent=2)

In [70]:
def run_prompt(test_case):
  """Merges the prompt and test case input, then returns the result"""
  prompt= f"""

Please solve the following task:

{test_case["task"]}

"""

  messages=[]
  add_user_message(messages, prompt)
  output = chat(messages)
  return output

def run_test_case(test_case):
  """Calls run_prompt, then grades the resullt"""
  output = run_prompt(test_case)

  # TODO -Grading
  score = 10

  return {

          "output": output,
          "test_case": test_case,
          "score": score
  }



def run_eval(dataset):
  """Loads the dataset and calls run_test_case for each test case"""
  results = []
  for test_case in dataset:
    result = run_test_case(test_case)
    results.append(result)

  return results

In [71]:
with open("dataset.json", "r") as f:
  dataset = json.load(f)

results = run_eval(dataset)

In [72]:
print(json.dumps(results, indent=2))

[
  {
    "output": "I'll write a Python function to parse an AWS ARN (Amazon Resource Name) string into its components.\n\n```python\ndef parse_arn(arn_string):\n    \"\"\"\n    Parse an AWS ARN string and return its components as a dictionary.\n    \n    ARN format: arn:partition:service:region:account-id:resource\n    \n    Args:\n        arn_string (str): The ARN string to parse\n        \n    Returns:\n        dict: A dictionary containing the ARN components\n        \n    Raises:\n        ValueError: If the ARN string is invalid\n    \"\"\"\n    if not arn_string:\n        raise ValueError(\"ARN string cannot be empty\")\n    \n    # Split the ARN by colons\n    parts = arn_string.split(':', 5)  # Split into max 6 parts\n    \n    # Validate that we have at least 6 parts\n    if len(parts) < 6:\n        raise ValueError(f\"Invalid ARN format. Expected at least 6 parts, got {len(parts)}\")\n    \n    # Validate that it starts with 'arn'\n    if parts[0] != 'arn':\n        raise Va